# Notebook 4 — Model Training and Comparison

Train and compare 11 different machine learning models on the flood prediction task.

## Models Tested
1. Random Forest
2. Extra Trees
3. XGBoost
4. LightGBM
5. Gradient Boosting
6. CatBoost
7. AdaBoost
8. Logistic Regression
9. Naive Bayes
10. Neural Network (MLP)
11. KNN

## Evaluation
- Temporal train-test split: 2019-2023 train, 2024 test
- Metrics: Precision, Recall, F1, ROC-AUC

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## 2. Load Data and Setup Features

In [ ]:
df = pd.read_csv("../data/processed/dhemaji_flood_FINAL.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

features = [
    "dist_to_major_river",
    "elevation",
    "tree_cover",
    "slope",
    "rain_anomaly",
    "rain_5day",
    "rain_3day",
    "rainfall_mm",
    "runoff_sum",
    "runoff_anomaly"
]

# Temporal split
train = df[df["year"] < 2024]
test  = df[df["year"] == 2024]

X_train = train[features]
y_train = train["flood_label"]
X_test  = test[features]
y_test  = test["flood_label"]

# Scaled version for Neural Network and KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Train years: 2019-2023")
print("Test year:   2024")

## 3. Define Main Models (Tree-Based and Linear)

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        min_samples_leaf=20,
        n_jobs=-1,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        scale_pos_weight=17,
        learning_rate=0.1,
        max_depth=6,
        n_jobs=-1,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        class_weight="balanced",
        learning_rate=0.1,
        max_depth=6,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        auto_class_weights="Balanced",
        random_state=42,
        verbose=0
    ),
    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.1,
        random_state=42
    ),
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        n_jobs=-1,
        random_state=42
    ),
    "Naive Bayes": GaussianNB()
}

## 4. Train and Evaluate Main Models

In [ ]:
results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    start = time.time()
    model.fit(X_train, y_train)
    train_time = round(time.time() - start, 1)
    
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    p   = precision_score(y_test, y_pred)
    r   = recall_score(y_test, y_pred)
    f   = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
        "ROC-AUC": round(auc, 3),
        "Time": f"{train_time}s"
    })
    
    print(f"  P:{p:.3f} R:{r:.3f} F1:{f:.3f} AUC:{auc:.3f} ({train_time}s)")

## 5. Train Neural Network (Requires Scaled Data)

In [ ]:
print("Training Neural Network...")

start = time.time()
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)
train_time = round(time.time() - start, 1)

y_pred  = mlp.predict(X_test_scaled)
y_proba = mlp.predict_proba(X_test_scaled)[:,1]

p   = precision_score(y_test, y_pred)
r   = recall_score(y_test, y_pred)
f   = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

results.append({
    "Model": "Neural Network",
    "Precision": round(p, 3),
    "Recall": round(r, 3),
    "F1": round(f, 3),
    "ROC-AUC": round(auc, 3),
    "Time": f"{train_time}s"
})

print(f"  P:{p:.3f} R:{r:.3f} F1:{f:.3f} AUC:{auc:.3f}")

## 6. Train KNN (Uses Small Sample — Slow on Full Data)

In [ ]:
print("Training KNN...")

# KNN is too slow on 1M rows — use sample
X_train_small = X_train.sample(50000, random_state=42)
y_train_small = y_train[X_train_small.index]
X_train_small_scaled = scaler.transform(X_train_small)

start = time.time()
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_small_scaled, y_train_small)
train_time = round(time.time() - start, 1)

y_pred  = knn.predict(X_test_scaled)
y_proba = knn.predict_proba(X_test_scaled)[:,1]

p   = precision_score(y_test, y_pred)
r   = recall_score(y_test, y_pred)
f   = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

results.append({
    "Model": "KNN",
    "Precision": round(p, 3),
    "Recall": round(r, 3),
    "F1": round(f, 3),
    "ROC-AUC": round(auc, 3),
    "Time": f"{train_time}s"
})

print(f"  P:{p:.3f} R:{r:.3f} F1:{f:.3f} AUC:{auc:.3f}")

## 7. Comparison Table

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    "F1", ascending=False
).reset_index(drop=True)

print("="*70)
print("COMPLETE MODEL COMPARISON")
print("="*70)
print(results_df.to_string(index=False))

results_df.to_csv("../results/model_comparison.csv", index=False)

## 8. Visualize Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(13,6))

model_short = {
    "Gradient Boosting":"GB",
    "Random Forest":"RF",
    "Neural Network":"NN",
    "Extra Trees":"ET",
    "XGBoost":"XGB",
    "LightGBM":"LGBM",
    "AdaBoost":"Ada",
    "CatBoost":"Cat",
    "KNN":"KNN",
    "Logistic Regression":"LR",
    "Naive Bayes":"NB"
}

labels = [model_short.get(m, m) for m in results_df["Model"]]

bars = ax.bar(
    labels, results_df["F1"],
    color="steelblue", edgecolor="black"
)

for bar, score in zip(bars, results_df["F1"]):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.01,
        f"{score}",
        ha="center", fontsize=10
    )

ax.set_title("Model Comparison — F1 Score")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("../figures/model_comparison.png", dpi=150)
plt.show()

## 9. Conclusions

**Gradient Boosting selected as the final model** based on:
- Highest F1 score (0.833)
- Best precision (0.845) — fewest false alarms
- Strong ROC-AUC (0.990)
- Acceptable training time for this problem size

Top 3 models all achieve F1 > 0.80:
- Gradient Boosting (0.833)
- Random Forest (0.823)
- Neural Network (0.812)

Models that did not perform well for this problem:
- KNN (0.598) — too slow on full data, used sample
- Logistic Regression (0.441) — linear model can't capture spatial patterns
- Naive Bayes (0.295) — feature independence assumption is wrong

Final model training is in Notebook 5.